# Assignment 5

Deadline: xx.xx.2026 12:00 CEST

## Task

Develop an investment strategy for the Swiss equity market, backtest it using the provided datasets (i.e., `market_data.parquet`, `jkp_data.parquet`, `spi_index.csv`) and analyze its performance by benchmarking it against the Swiss Performance Index (SPI). Build on the existing qpmwp-course codebase and extend it with any additional components required. Summarize your approach and findings in a written report.

### Coding (15 points)



- Selection:
  Use some of the existing selection item builder functions (in `bibfn_selection.py`, applied via `SelectionItemBuilder`) to filter stocks based on specific criteria, and/or implement your own filters (e.g., exclude low-quality or high-volatility stocks).

- Optimization Data & Constraints:
  Use the implemented functions for preparing optimization inputs (in `bibfn_optimization_data.py`, applied via `OptimizationItemBuilder`) and for specifying constraints such as stock, sector, or factor exposure limits (in `bibfn_constraints.py`, applied via `OptimizationItemBuilder`). Extend these with your own functions where appropriate.

- Signal Generation:
  Use a machine learning method to estimate optimization inputs such as expected returns or risk. Possible approaches include regression, classification, or learning-to-rank models. A good starting point is the demo notebook `xsection_regressor.ipynb` which shows how to generate predictive signals. Use jkp_data as features or engineer your own (e.g., technical indicators from returns or prices). Alternatively, or in combination, you may use a factor model to construct the optimization inputs.

- Optimization Model:
  Use an optimization model to determine portfolio weights. You may rely on an existing class (e.g., `MeanVariance`, `LeastSquares`, or `BlackLitterman`) or implement a custom model. If you choose to create a custom optimization model, develop a class inheriting from `Optimization` and ensure implement the methods `set_objective` (to construct the coefficients of the objective function) and `solve` (to run the optimization).

- Simulation:
  Backtest the strategy and simulate portfolio returns, incorporating fixed costs of 1% per year and transaction costs of 0.2% per rebalancing.


### Report (15 points):

Produce an HTML report (for example, by converting a .ipynb notebook to HTML) containing:

- High-level strategy overview: A clear description of the investment strategy and its rationale.

- Detailed explanation of the backtesting steps: A step-by-step explanation of the backtest design, including model choices and implementation details (e.g., the machine learning method or factor model used).

- Backtesting results:
    
    - Charts: Visualizations such as cumulative performance, rolling 3-year returns, etc.
    - Descriptive statistics: Key statistics such as mean, standard deviation, drawdown, turnover, and Sharpe ratio (or any other relevant metric) for the full backtest period as well as for subperiods (e.g., the last 5 years, or during bull vs. bear market phases).
    - Compare your strategy against the SPI.


In [1]:

# Strategy overview
# -----------------
# 1. Load Swiss equity data: market_data.parquet, jkp_data.parquet, spi_index.csv.
# 2. Build custom technical factors:
#       - medium-term momentum: 12-month return excluding the last month;
#       - short-term reversal: negative 1-month return;
#       - realized volatility: 3-month daily return volatility.
# 3. Combine them with JKP quality/fundamental factors.
# 4. Train an XGBoost learn-to-rank model cross-sectionally to predict future
#    3-month return rankings.
# 5. Convert model scores into Black-Litterman views.
# 6. Optimize a long-only portfolio with liquidity, trading-gap, concentration,
#    size-dependent and turnover constraints.
# 7. Backtest versus the Swiss Performance Index including fixed and transaction
#    costs.



import os
import sys
import warnings
from dataclasses import dataclass
from typing import Iterable, Optional


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("Please install xgboost: pip install xgboost") from exc


project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course\\src')
sys.path.append(project_root)
sys.path.append(src_path)

import os
import sys

PROJECT_ROOT = "/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course"
PATH_TO_DATA = "/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/data"
SAVE_PATH = "/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/results"

SRC_PATH = os.path.join(PROJECT_ROOT, "src")

sys.path.append(PROJECT_ROOT)
sys.path.append(SRC_PATH)



from helper_functions import load_data_spi, align_market_data_with_jkp_data
from estimation.covariance import Covariance
from optimization.optimization import BlackLitterman
from backtesting.backtest_item_builder.bib_classes import (
    SelectionItemBuilder,
    OptimizationItemBuilder,
)
from backtesting.backtest_item_builder.bibfn_selection import (
    bibfn_selection_gaps,
    bibfn_selection_min_volume,
    bibfn_selection_jkp_data_scores,
)
from backtesting.backtest_item_builder.bibfn_optimization_data import (
    bibfn_return_series,
    bibfn_scores,
    bibfn_cap_weights,
)
from backtesting.backtest_item_builder.bibfn_constraints import (
    bibfn_budget_constraint,
    bibfn_box_constraints,
    bibfn_size_dependent_upper_bounds,
    bibfn_turnover_constraint,
)
from backtesting.backtest_data import BacktestData
from backtesting.backtest_service import BacktestService
from backtesting.backtest import Backtest

warnings.filterwarnings("ignore", category=RuntimeWarning)



@dataclass
class StrategyConfig:

    path_to_data: str = PATH_TO_DATA
    save_path: str = SAVE_PATH

 
    start_date: str = "2003-01-31"
    end_date: Optional[str] = "2022-12-31"
    rebalance_months: int = 3
    lookback_days: int = 365 * 3

   
    min_volume: float = 500_000
    volume_window_days: int = 365
    max_zero_volume_gap_days: int = 10

   
    forward_return_months: int = 3
    min_train_dates: int = 24

    
    fixed_costs_annual: float = 0.01
    transaction_costs_per_rebalancing: float = 0.002

   
    stock_upper_bound: float = 0.10
    turnover_limit: float = 0.15

    tau_psi: Optional[float] = 0.02
    tau_omega: Optional[float] = 0.05
    view_gen_algo: str = "quintile_sort"
    solver_name: str = "cvxopt"

    
    jkp_quality_features: tuple[str, ...] = (
    "qmj",
    "value",
    "investment",
    )
    custom_features: tuple[str, ...] = (
    "momentum",
    "reversal_1w",
    "low_vol",
    "mom_vol_adj",
)




def ensure_directory(path: str) -> None:
    if path and not os.path.exists(path):
        os.makedirs(path, exist_ok=True)


def winsorize_cross_section(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """Winsorize a cross-section to reduce the influence of extreme values."""
    if s.dropna().empty:
        return s
    lo, hi = s.quantile([lower, upper])
    return s.clip(lower=lo, upper=hi)


def zscore_cross_section(s: pd.Series) -> pd.Series:
    """Cross-sectional z-score for a single date."""
    std = s.std(skipna=True)
    if pd.isna(std) or std == 0:
        return s * 0
    return (s - s.mean(skipna=True)) / std


def prepare_factor_panel(df: pd.DataFrame, feature_cols: Iterable[str]) -> pd.DataFrame:
    """
    Winsorize and z-score all features date-by-date.

    Expected index: MultiIndex(date, id).
    """
    out = df.copy()
    for col in feature_cols:
        out[col] = (
            out[col]
            .groupby(level="date", group_keys=False)
            .apply(winsorize_cross_section)
            .groupby(level="date", group_keys=False)
            .apply(zscore_cross_section)
        )
    return out


def get_price_or_return_series(market_data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Return wide price and daily return matrices from market_data.
    Robust to duplicate (date, id) rows.
    """

    md = market_data.copy().sort_index()

    # Remove duplicate date-id observations before unstack
    if md.index.duplicated().any():
        print(f"Warning: dropping {md.index.duplicated().sum()} duplicate market_data rows.")
        md = md[~md.index.duplicated(keep="last")]

    price_candidates = ["price", "prc", "close", "adj_close"]
    return_candidates = ["ret", "return", "returns"]

    price_col = next((col for col in price_candidates if col in md.columns), None)
    return_col = next((col for col in return_candidates if col in md.columns), None)

    prices = None
    daily_returns = None

    if price_col is not None:
        prices = md[price_col].unstack("id").sort_index()
        daily_returns = prices.pct_change()

    if return_col is not None:
        daily_returns = md[return_col].unstack("id").sort_index()
        if prices is None:
            prices = (1 + daily_returns.fillna(0)).cumprod()

    if prices is None or daily_returns is None:
        raise ValueError(
            f"Could not infer prices or returns. Available columns: {list(md.columns)}"
        )

    return prices, daily_returns


def build_custom_technical_factors(
    market_data: pd.DataFrame,
    jkp_dates: pd.DatetimeIndex,
) -> pd.DataFrame:

    md = market_data.copy().sort_index()

    if md.index.duplicated().any():
        md = md[~md.index.duplicated(keep="last")]

    price_col = next(
        col for col in ["price", "prc", "close", "adj_close"]
        if col in md.columns
    )

    prices = md[price_col].unstack("id").sort_index()

    # Daily returns
    daily_ret = prices.pct_change()


    # smoother 6M momentum
    mom_6m = prices.pct_change(126)

    # 12M excluding last month
    mom_12m = prices.pct_change(252)
    mom_1m = prices.pct_change(21)

    mom_12_1 = (1 + mom_12m) / (1 + mom_1m) - 1

    # combine both signals
    momentum = 0.5 * mom_6m + 0.5 * mom_12_1

   
    # use weekly reversal instead of noisy monthly reversal
    reversal_1w = -prices.pct_change(5)

    # smooth cross-sectionally
    reversal_1w = reversal_1w.rolling(5).mean()


    vol_3m = daily_ret.rolling(63).std()

    # prefer low volatility stocks
    low_vol = -vol_3m


    momentum_vol_adj = momentum / (vol_3m + 1e-6)

    factors = {}

    factor_dict = {
        "momentum": momentum,
        "reversal_1w": reversal_1w,
        "low_vol": low_vol,
        "mom_vol_adj": momentum_vol_adj,
    }

    for name, df in factor_dict.items():

        aligned = (
            df.reindex(df.index.union(jkp_dates))
            .sort_index()
            .ffill()
            .loc[jkp_dates]
        )

        factors[name] = aligned.stack()

    custom_factors = pd.concat(factors, axis=1)
    custom_factors.index.names = ["date", "id"]

    # robust normalization
    for col in custom_factors.columns:

        custom_factors[col] = (
            custom_factors[col]
            .groupby(level="date")
            .transform(
                lambda x:
                (x.clip(x.quantile(0.05), x.quantile(0.95)) - x.mean())
                / (x.std() + 1e-8)
            )
        )

    return custom_factors

def build_ml_dataset(
    market_data: pd.DataFrame,
    jkp_data: pd.DataFrame,
    config: StrategyConfig,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Combine JKP quality/fundamental factors with custom technical factors and
    create the forward-return target for learn-to-rank.
    """
    jkp_data = jkp_data.sort_index()
    jkp_dates = jkp_data.index.get_level_values("date").unique().sort_values()

    available_jkp_features = [c for c in config.jkp_quality_features if c in jkp_data.columns]
    missing = sorted(set(config.jkp_quality_features) - set(available_jkp_features))
    if missing:
        print(f"Warning: these requested JKP features are missing and will be skipped: {missing}")

    custom_factors = build_custom_technical_factors(market_data=market_data, jkp_dates=jkp_dates)

    feature_panel = jkp_data[available_jkp_features].join(custom_factors, how="left")
    feature_cols = available_jkp_features + list(config.custom_features)
    feature_cols = [c for c in feature_cols if c in feature_panel.columns]

    feature_panel = prepare_factor_panel(feature_panel, feature_cols)

    # Forward return target from daily returns over the next rebalance horizon.
    _, daily_returns = get_price_or_return_series(market_data)
    daily_returns = daily_returns.sort_index()
    horizon_days = int(config.forward_return_months * 21)
    fwd_return = (1 + daily_returns).rolling(horizon_days, min_periods=max(10, horizon_days // 2)).apply(
        np.prod,
        raw=True,
    ) - 1
    fwd_return = fwd_return.shift(-horizon_days)

    # Align target to JKP dates.
    target_rows = []
    market_dates = fwd_return.index.sort_values()
    for date in jkp_dates:
        eligible = market_dates[market_dates <= date]
        if len(eligible) == 0:
            continue
        last_market_date = eligible[-1]
        row = fwd_return.loc[last_market_date].copy()
        row.name = pd.to_datetime(date)
        target_rows.append(row)

    target_wide = pd.DataFrame(target_rows)
    target_wide.index.name = "date"
    target = target_wide.stack(dropna=False).rename("fwd_ret_3m").to_frame()
    target.index.names = ["date", "id"]

    dataset = feature_panel.join(target, how="left")
    dataset = dataset.replace([np.inf, -np.inf], np.nan)

    return dataset, feature_cols


def add_ranking_target(dataset: pd.DataFrame, target_col: str = "fwd_ret_3m") -> pd.DataFrame:
    """
    Convert future returns into date-wise ranking labels.

    XGBoost rankers require non-negative relevance labels. We use decile labels
    from 0 to 9 within each date, where 9 is the best future return bucket.
    """
    df = dataset.copy()

    def rank_to_decile(s: pd.Series) -> pd.Series:
        if s.dropna().nunique() < 2:
            return pd.Series(np.nan, index=s.index)
        ranks = s.rank(method="first", ascending=True)
        labels = pd.qcut(ranks, q=10, labels=False, duplicates="drop")
        return labels.astype(float)

    df["rank_label"] = df.groupby(level="date")[target_col].transform(rank_to_decile)
    return df


def train_xgb_ltr_and_predict(
    dataset: pd.DataFrame,
    feature_cols: list[str],
    config: StrategyConfig,
) -> pd.Series:
    """
    Expanding-window cross-sectional XGBoost learn-to-rank.

    At each rebalancing date t, the model is trained only on dates strictly before t.
    Predictions at t become the out-of-sample ml_signal.
    """
    df = add_ranking_target(dataset)
    dates = df.index.get_level_values("date").unique().sort_values()

    predictions = []

    for i, pred_date in enumerate(dates):
        train_dates = dates[:i]
        if len(train_dates) < config.min_train_dates:
            continue

        train = df.loc[df.index.get_level_values("date").isin(train_dates)]
        test = df.loc[df.index.get_level_values("date") == pred_date]

        train = train.dropna(subset=feature_cols + ["rank_label"])
        test = test.dropna(subset=feature_cols)

        if train.empty or test.empty:
            continue

        # XGBoost ranking groups: number of stocks per date.
        train_dates_clean = train.index.get_level_values("date")
        group_sizes = train.groupby(train_dates_clean).size().to_numpy()

        X_train = train[feature_cols].astype(float)
        y_train = train["rank_label"].astype(float)
        X_test = test[feature_cols].astype(float)

        model = xgb.XGBRanker(
            objective="rank:pairwise",
            eval_metric="ndcg",
            n_estimators=150,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            reg_alpha=0.10,
            reg_lambda=1.00,
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
        )

        model.fit(X_train, y_train, group=group_sizes, verbose=False)
        pred = pd.Series(model.predict(X_test), index=test.index, name="ml_signal")
        predictions.append(pred)

        if i % 12 == 0:
            print(f"Generated ML signal for {pd.to_datetime(pred_date).date()} ({i + 1}/{len(dates)})")

    if not predictions:
        raise RuntimeError("No ML predictions were generated. Check data availability and min_train_dates.")

    ml_signal = pd.concat(predictions).sort_index()
    ml_signal = (
        ml_signal
        .groupby(level="date", group_keys=False)
        .apply(winsorize_cross_section)
        .groupby(level="date", group_keys=False)
        .apply(zscore_cross_section)
    )
    ml_signal.name = "ml_signal"
    return ml_signal




def load_and_prepare_data(config: StrategyConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    market_data = pd.read_parquet(os.path.join(config.path_to_data, "market_data.parquet"))
    jkp_data = pd.read_parquet(os.path.join(config.path_to_data, "jkp_data.parquet"))

    # SPI loader is used in the course examples. It usually expects the data directory.
    spi = load_data_spi(path=config.path_to_data)

    # Ensure date levels are timestamps.
    if isinstance(market_data.index, pd.MultiIndex):
        market_data = market_data.copy()
        market_data.index = market_data.index.set_levels(
            pd.to_datetime(market_data.index.levels[0]),
            level="date",
        )
    if isinstance(jkp_data.index, pd.MultiIndex):
        jkp_data = jkp_data.copy()
        jkp_data.index = jkp_data.index.set_levels(
            pd.to_datetime(jkp_data.index.levels[0]),
            level="date",
        )

    return market_data, jkp_data, spi


def build_rebalancing_dates(
    market_data: pd.DataFrame,
    jkp_data: pd.DataFrame,
    config: StrategyConfig,
) -> list[str]:
    market_dates = market_data.index.get_level_values("date").unique().sort_values()
    jkp_dates = jkp_data.index.get_level_values("date").unique().sort_values()

    rebdates = (
        jkp_dates[jkp_dates > market_dates.min()][:: config.rebalance_months]
        .strftime("%Y-%m-%d")
        .tolist()
    )
    rebdates = [d for d in rebdates if d >= config.start_date]
    if config.end_date is not None:
        rebdates = [d for d in rebdates if d <= config.end_date]
    # Drop last rebalance date to ensure there is an out-of-sample holding period.
    if len(rebdates) > 1:
        rebdates = rebdates[:-1]
    return rebdates


def build_backtest_service(
    market_data_ffill: pd.DataFrame,
    jkp_data_with_signal: pd.DataFrame,
    spi: pd.Series,
    rebdates: list[str],
    config: StrategyConfig,
) -> BacktestService:
    data = BacktestData()
    data.market_data = market_data_ffill
    data.jkp_data = jkp_data_with_signal
    data.bm_series = spi

    selection_item_builders = {
        "gaps": SelectionItemBuilder(
            bibfn=bibfn_selection_gaps,
            width=config.lookback_days,
            n_days=config.max_zero_volume_gap_days,
        ),
        "min_volume": SelectionItemBuilder(
            bibfn=bibfn_selection_min_volume,
            width=config.volume_window_days,
            min_volume=config.min_volume,
            agg_fn=np.median,
        ),
        "jkp_data_scores": SelectionItemBuilder(
            bibfn=bibfn_selection_jkp_data_scores,
            fields=["ml_signal"],
        ),
    }

    optimization_item_builders = {
        "return_series": OptimizationItemBuilder(
            bibfn=bibfn_return_series,
            width=config.lookback_days,
            weekdays_only=True,
            fillna_value=0,
        ),
        "cap_weights": OptimizationItemBuilder(
            bibfn=bibfn_cap_weights,
        ),
        "scores": OptimizationItemBuilder(
            bibfn=bibfn_scores,
            fields=["ml_signal"],
        ),
        "budget_constraint": OptimizationItemBuilder(
            bibfn=bibfn_budget_constraint,
            budget=1,
        ),
        "box_constraints": OptimizationItemBuilder(
            bibfn=bibfn_box_constraints,
            lower=0,
            upper=config.stock_upper_bound,
        ),
        "size_dep_upper_bounds": OptimizationItemBuilder(
            bibfn=bibfn_size_dependent_upper_bounds,
            small_cap={"threshold": 300_000_000, "upper": 0.02},
            mid_cap={"threshold": 1_000_000_000, "upper": 0.05},
            large_cap={"threshold": 10_000_000_000, "upper": config.stock_upper_bound},
        ),
        "turnover_constraint": OptimizationItemBuilder(
            bibfn=bibfn_turnover_constraint,
            turnover_limit=config.turnover_limit,
        ),
    }

    tau_default = 1 / config.lookback_days

    optimization = BlackLitterman(
        covariance=Covariance(method="pearson", check_positive_definite=True),
        tau_psi=config.tau_psi or tau_default,
        tau_omega=config.tau_omega or tau_default,
        view_gen_algo=config.view_gen_algo,
        use_unconditional_cov=True,
        signal_names=["ml_signal"],
        solver_name=config.solver_name,
    )

    bs = BacktestService(
        data=data,
        optimization=optimization,
        selection_item_builders=selection_item_builders,
        optimization_item_builders=optimization_item_builders,
        rebdates=rebdates,
        quiet=False,
    )
    return bs



def sim_outperformance(x: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    return x.subtract(y, axis=0).divide(1 + y, axis=0)

def performance_table(sim: pd.DataFrame, benchmark_col: str = "Benchmark") -> pd.DataFrame:
    metrics = {}

    periods_per_year = 252

    for col in sim.columns:
        r = sim[col].dropna()
        common = sim[[col, benchmark_col]].dropna()

        cumulative_return = (1 + r).prod() - 1
        n_periods = len(r)
        annual_return = (1 + cumulative_return) ** (periods_per_year / n_periods) - 1
        annual_volatility = r.std() * np.sqrt(periods_per_year)

        sharpe_ratio = (
            annual_return / annual_volatility
            if annual_volatility != 0
            else np.nan
        )

        wealth = (1 + r).cumprod()
        running_max = wealth.cummax()
        drawdown = wealth / running_max - 1
        max_drawdown = drawdown.min()

        if col != benchmark_col:
            active_return = common[col] - common[benchmark_col]
            tracking_error = active_return.std() * np.sqrt(periods_per_year)

            x = common[benchmark_col].values
            y = common[col].values

            beta = np.cov(y, x)[0, 1] / np.var(x) if np.var(x) != 0 else np.nan
            alpha_daily = y.mean() - beta * x.mean()
            alpha = alpha_daily * periods_per_year
        else:
            tracking_error = 0.0
            alpha = 0.0
            beta = 1.0

        metrics[col] = {
            "Annual Return": annual_return,
            "Cumulative Return": cumulative_return,
            "Annual Volatility": annual_volatility,
            "Sharpe Ratio": sharpe_ratio,
            "Max Drawdown": max_drawdown,
            "Tracking Error": tracking_error,
            "Alpha": alpha,
            "Beta": beta,
        }

    return pd.DataFrame(metrics).T



def plot_results(sim: pd.DataFrame, turnover: pd.Series, save_path: str) -> None:
    ensure_directory(save_path)

    ax = np.log1p(sim).cumsum().plot(figsize=(11, 6), title="Cumulative Log Performance")
    ax.set_ylabel("Cumulative log return")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "cumulative_log_performance.png"), dpi=150)
    plt.close()

    sim_rel = sim_outperformance(sim, sim["Benchmark"])
    ax = np.log1p(sim_rel).cumsum().plot(figsize=(11, 6), title="Cumulative Out-/Underperformance vs SPI")
    ax.set_ylabel("Cumulative relative log return")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "relative_performance_vs_spi.png"), dpi=150)
    plt.close()

    ax = turnover.plot(figsize=(11, 5), title="Portfolio Turnover")
    ax.set_ylabel("Turnover")
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "turnover.png"), dpi=150)
    plt.close()




def main() -> None:
    config = StrategyConfig()
    ensure_directory(config.save_path)

    print("Loading data...")
    market_data, jkp_data, spi = load_and_prepare_data(config)

    print("Building ML dataset and custom factors...")
    ml_dataset, feature_cols = build_ml_dataset(
        market_data=market_data,
        jkp_data=jkp_data,
        config=config,
    )
    print(f"Features used by XGBoost LTR: {feature_cols}")

    print("Training XGBoost Learn-to-Rank and generating out-of-sample signal...")
    ml_signal = train_xgb_ltr_and_predict(
        dataset=ml_dataset,
        feature_cols=feature_cols,
        config=config,
    )
    ml_signal.to_frame().to_parquet(os.path.join(config.save_path, "ltr_signal.parquet"))

    print("Adding ML signal to jkp_data...")
    jkp_data_with_signal = jkp_data.join(ml_signal.to_frame(), how="left")

    print("Aligning market_data and jkp_data...")
    market_data_ffill, jkp_data_with_signal = align_market_data_with_jkp_data(
        market_data=market_data,
        jkp_data=jkp_data_with_signal,
    )

    print("Building rebalancing dates...")
    rebdates = build_rebalancing_dates(
        market_data=market_data_ffill,
        jkp_data=jkp_data_with_signal,
        config=config,
    )
    print(f"Number of rebalancing dates: {len(rebdates)}")
    print(f"First rebalance: {rebdates[0]}, last rebalance: {rebdates[-1]}")

    print("Building backtest service...")
    bs = build_backtest_service(
        market_data_ffill=market_data_ffill,
        jkp_data_with_signal=jkp_data_with_signal,
        spi=spi,
        rebdates=rebdates,
        config=config,
    )

    print("Running Black-Litterman backtest...")
    bt_bl = Backtest()
    bt_bl.run(bs=bs)
    bt_bl.save(path=config.save_path, filename="bt_ml_black_litterman.pickle")

    print("Simulating gross and net performance...")
    return_series = bs.data.get_return_series(weekdays_only=False)

    sim_bl_gross = bt_bl.strategy.simulate(
        return_series=return_series,
        fc=0,
        vc=0,
    )
    sim_bl_net = bt_bl.strategy.simulate(
        return_series=return_series,
        fc=config.fixed_costs_annual,
        vc=config.transaction_costs_per_rebalancing,
    )

    sim = pd.concat(
        {
            "Benchmark": bs.data.bm_series,
            "ML-BL Gross": sim_bl_gross,
            "ML-BL Net": sim_bl_net,
        },
        axis=1,
    ).dropna()

    if config.end_date is not None:
        sim = sim[sim.index <= config.end_date]

    sim.to_csv(os.path.join(config.save_path, "simulation_returns.csv"))

    print("Computing turnover and performance metrics...")
    turnover = bt_bl.strategy.turnover(return_series=return_series)
    turnover.to_csv(os.path.join(config.save_path, "turnover.csv"), header=["turnover"])

    stats = performance_table(sim, benchmark_col="Benchmark")
    stats.to_csv(os.path.join(config.save_path, "performance_metrics.csv"))
    print("\nPerformance metrics:")
    print(stats.round(4))

    print("Creating plots...")
    plot_results(sim=sim, turnover=turnover, save_path=config.save_path)

    print("\nDone. Outputs saved in:")
    print(os.path.abspath(config.save_path))


if __name__ == "__main__":
    main()

Loading data...
Building ML dataset and custom factors...


/var/folders/5p/qkrf3s2s08q9rspfsbwy6_kw0000gn/T/ipykernel_52783/1115127161.py:233: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_ret = prices.pct_change()
/var/folders/5p/qkrf3s2s08q9rspfsbwy6_kw0000gn/T/ipykernel_52783/1115127161.py:237: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  mom_6m = prices.pct_change(126)
/var/folders/5p/qkrf3s2s08q9rspfsbwy6_kw0000gn/T/ipykernel_52783/1115127161.py:240: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_ch

/var/folders/5p/qkrf3s2s08q9rspfsbwy6_kw0000gn/T/ipykernel_52783/1115127161.py:200: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_returns = prices.pct_change()
/var/folders/5p/qkrf3s2s08q9rspfsbwy6_kw0000gn/T/ipykernel_52783/1115127161.py:351: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  target = target_wide.stack(dropna=False).rename("fwd_ret_3m").to_frame()


Features used by XGBoost LTR: ['qmj', 'momentum', 'reversal_1w', 'low_vol', 'mom_vol_adj']
Training XGBoost Learn-to-Rank and generating out-of-sample signal...
Generated ML signal for 1994-12-31 (109/460)
Generated ML signal for 1995-12-31 (121/460)
Generated ML signal for 1996-12-31 (133/460)
Generated ML signal for 1997-12-31 (145/460)
Generated ML signal for 1998-12-31 (157/460)
Generated ML signal for 1999-12-31 (169/460)
Generated ML signal for 2000-12-31 (181/460)
Generated ML signal for 2001-12-31 (193/460)
Generated ML signal for 2002-12-31 (205/460)
Generated ML signal for 2003-12-31 (217/460)
Generated ML signal for 2004-12-31 (229/460)
Generated ML signal for 2005-12-31 (241/460)
Generated ML signal for 2006-12-31 (253/460)
Generated ML signal for 2007-12-31 (265/460)
Generated ML signal for 2008-12-31 (277/460)
Generated ML signal for 2009-12-31 (289/460)
Generated ML signal for 2010-12-31 (301/460)
Generated ML signal for 2011-12-31 (313/460)
Generated ML signal for 2012-

/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/src/backtesting/strategy.py:195: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  portf_ret[0] -= varcost[0]
/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/src/backtesting/strategy.py:195: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  portf_ret[0] -= varcost[0]


Computing turnover and performance metrics...

Performance metrics:
             Annual Return  Cumulative Return  Annual Volatility  \
Benchmark           0.0741             3.4454             0.1607   
ML-BL Gross         0.0858             4.5688             0.1140   
ML-BL Net           0.0695             3.0626             0.1140   

             Sharpe Ratio  Max Drawdown  Tracking Error   Alpha    Beta  
Benchmark          0.4612       -0.5325          0.0000  0.0000  1.0000  
ML-BL Gross        0.7527       -0.4302          0.0721  0.0339  0.6508  
ML-BL Net          0.6098       -0.4460          0.0722  0.0187  0.6508  
Creating plots...

Done. Outputs saved in:
/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/results
